In [ ]:
# ==============================================================================
# INSTALASI DEPENDENSI (Jalankan sel ini HANYA jika pustaka belum terpasang)
# Hilangkan tanda pagar (#) pada baris di bawah untuk memasang dependensi:
# ==============================================================================
# !pip install numpy matplotlib scikit-learn quantile-forest
# Atau instal seluruh dependensi modul melalui file requirements.txt:
# !pip install -r ../requirements.txt


# PRAKTIK 5: MACHINE LEARNING UNTUK DIGITAL SOIL MAPPING (DSM)
## Pemodelan Spasial SCORPAN, Random Forest, dan Ketidakpastian C-Organik Tanah (0–20 cm)
**Pelatihan Pemanfaatan RS dan DSM untuk Pertanian Presisi**  
Pusat Studi Pengelolaan Sumber Daya Lahan (PS2DL) UGM & PT SMART Tbk

---
> **Petunjuk Lingkungan Kerja (Virtual Environment / venv):**  
> Pastikan notebook ini dijalankan pada kernel virtual environment **`Python (Pelatihan SMART)`** (`env_smart`).  
> Jika belum membuat venv, jalankan skrip penyiapan otomatis:  
> - **Linux / macOS:** `./setup_env.sh` (pada folder `Output/Pelatihan/`)  
> - **Windows:** `setup_env.bat` (pada folder `Output\Pelatihan\`)  
> Periksa pojok kanan atas jendela JupyterLab; pastikan kernel yang aktif tertulis `Python (Pelatihan SMART)`.

---
### Tujuan Praktikum:
1. Menumpangkan 50 koordinat titik sampel lapangan ke tumpukan raster kovariat SCORPAN (*Point-in-Raster Overlay*).
2. Menganalisis peringkat kepentingan variabel (*Feature Importance*) dalam mengendalikan sebaran spasial C-Organik.
3. Melatih model *Random Forest Regressor* dan mengevaluasi akurasi menggunakan 5-Fold Cross-Validation.
4. Memprediksi nilai kontinu C-Organik (Lapisan 0–20 cm) ke seluruh kisi Sub-DAS Bompon dan mengekspor GeoTIFF.
5. Memodelkan ketidakpastian prediksi spasial menggunakan *Quantile Regression Forest* (QRF) pada interval keyakinan 90% (PI90).

In [ ]:
# 1. Inisialisasi Pustaka Python
import os
import csv
import numpy as np
import matplotlib.pyplot as plt
from osgeo import gdal
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error
from quantile_forest import RandomForestQuantileRegressor

gdal.UseExceptions()
print('Pustaka Machine Learning dan Spasial berhasil dimuat!')

### 2. Memuat Data Titik Sampel Lapangan dan Hasil Lab C-Organik Tanah
Kita membaca berkas `sampel_tanah_50titik.csv` dengan fokus target pada kolom `C_Organik_1` (kadar C-Organik lapisan 0–20 cm).

In [ ]:
# 2. Membaca data uji lab 50 titik sampel
csv_path = '../data/praktik-5/sampel_tanah_50titik.csv'
if not os.path.exists(csv_path):
    csv_path = 'Output/Pelatihan/data/praktik-5/sampel_tanah_50titik.csv'

points_data = []
with open(csv_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for r in reader:
        points_data.append({
            'id': r['Titik'],
            'x': float(r['x']),
            'y': float(r['y']),
            'c_org': float(r['C_Organik_1']), # Target: C-Organik 0-20 cm
            'clay': float(r['Lempung_1']),
            'sand': float(r['Pasir_1']),
            'ph': float(r['pH_H2O_1'])
        })

print(f'Total titik sampel tanah: {len(points_data)} titik')
print(f'Target Prediksi         : Kadar C-Organik Lapisan 1 (0-20 cm)')
print(f'Rata-rata C-Organik     : {np.mean([p["c_org"] for p in points_data]):.2f}% (Std: {np.std([p["c_org"] for p in points_data]):.2f}%)')

### 3. Ekstraksi Nilai Kovariat SCORPAN di Titik Sampel (Point Intersect)
Kita mengekstrak nilai piksel 11 kovariat lingkungan SCORPAN pada koordinat 50 titik sampel.

In [ ]:
# 3. Ekstraksi Nilai Kovariat Raster di Titik Koordinat
kovariat_dir = '../data/praktik-5/kovariat'
if not os.path.exists(kovariat_dir):
    kovariat_dir = 'Output/Pelatihan/data/praktik-5/kovariat'

raster_files = {
    'dem': 'dem_10m.tif',
    'slope': 'slope_10m.tif',
    'aspect': 'aspect_10m.tif',
    'hillshade': 'hillshade_10m.tif',
    'tpi': 'tpi_10m.tif',
    'pl': 'penggunaan_lahan_10m.tif',
    'sar_vv': ('sentinel1_sar_10m.tif', 1),
    'sar_vh': ('sentinel1_sar_10m.tif', 2),
    's2_b02': ('sentinel2_l2a_10m.tif', 1),
    's2_b04': ('sentinel2_l2a_10m.tif', 3),
    's2_b08': ('sentinel2_l2a_10m.tif', 7),
    's2_b11': ('sentinel2_l2a_10m.tif', 9)
}

ref_ds = gdal.Open(os.path.join(kovariat_dir, 'dem_10m.tif'))
gt = ref_ds.GetGeoTransform()
inv_gt = gdal.InvGeoTransform(gt)

cov_arrays = {}
for name, item in raster_files.items():
    fname, b_idx = item if isinstance(item, tuple) else (item, 1)
    ds = gdal.Open(os.path.join(kovariat_dir, fname))
    cov_arrays[name] = ds.GetRasterBand(b_idx).ReadAsArray().astype('float32')

# Turunkan NDVI dan BSI
b08 = cov_arrays['s2_b08']
b04 = cov_arrays['s2_b04']
b11 = cov_arrays['s2_b11']
b02 = cov_arrays['s2_b02']
cov_arrays['ndvi'] = (b08 - b04) / (b08 + b04 + 1e-6)
cov_arrays['bsi'] = ((b11 + b04) - (b08 + b02)) / ((b11 + b04) + (b08 + b02) + 1e-6)

cov_names = ['dem', 'slope', 'aspect', 'hillshade', 'tpi', 'sar_vv', 'sar_vh', 'ndvi', 'bsi', 's2_b11', 'pl']
X_list = []
y_list = []
coords = []

for pt in points_data:
    px, py = gdal.ApplyGeoTransform(inv_gt, pt['x'], pt['y'])
    px, py = int(round(px)), int(round(py))
    if 0 <= py < cov_arrays['dem'].shape[0] and 0 <= px < cov_arrays['dem'].shape[1]:
        vals = [cov_arrays[k][py, px] for k in cov_names]
        X_list.append(vals)
        y_list.append(pt['c_org'])
        coords.append((pt['x'], pt['y']))

X = np.array(X_list)
y = np.array(y_list)
coords = np.array(coords)
print(f'Ekstraksi berhasil: {X.shape[0]} titik x {X.shape[1]} kovariat SCORPAN')
print('Kovariat terpilih:', cov_names)

### 4. Pelatihan Model Random Forest & Validasi Silang C-Organik
Kita melatih algoritma *Random Forest Regressor* dan menghitung metrik akurasi menggunakan 5-Fold Cross-Validation.

In [ ]:
# 4. Pelatihan Random Forest dan Evaluasi K-Fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = np.zeros_like(y)

for train_idx, val_idx in kf.split(X):
    rf_cv = RandomForestRegressor(n_estimators=300, max_depth=5, random_state=42)
    rf_cv.fit(X[train_idx], y[train_idx])
    y_pred_cv[val_idx] = rf_cv.predict(X[val_idx])

r2_cv = r2_score(y, y_pred_cv)
rmse_cv = root_mean_squared_error(y, y_pred_cv)
mae_cv = mean_absolute_error(y, y_pred_cv)

print('=== HASIL EVALUASI 5-FOLD CROSS-VALIDATION C-ORGANIK ===')
print(f'Koefisien Determinasi (R²) : {r2_cv:.3f}')
print(f'Root Mean Squared Error    : {rmse_cv:.3f}%')
print(f'Mean Absolute Error (MAE)  : {mae_cv:.3f}%')

# Latih model final pada seluruh data 50 titik
rf_final = RandomForestRegressor(n_estimators=300, max_depth=5, random_state=42)
rf_final.fit(X, y)

# Peringkat Kepentingan Fitur (Feature Importance)
importances = rf_final.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
print('\nPeringkat Kepentingan Kovariat SCORPAN untuk C-Organik:')
for rank, idx in enumerate(sorted_idx, 1):
    print(f'{rank}. {cov_names[idx]:10s} : {importances[idx]*100:.1f}%')

### 5. Prediksi Spasial ke Seluruh Grid Sub-DAS Bompon
Model Random Forest diterapkan ke setiap piksel di dalam batas Sub-DAS Bompon untuk mengekspor peta kontinu C-Organik ke GeoTIFF.

In [ ]:
# 5. Prediksi Spasial Seluruh Piksel dan Ekspor GeoTIFF C-Organik
rows, cols = cov_arrays['dem'].shape
X_full = np.column_stack([cov_arrays[k].ravel() for k in cov_names])

# Hanya prediksi pada piksel di dalam Sub-DAS Bompon (dem > 0 dan tidak NaN)
valid_mask = ~np.isnan(X_full).any(axis=1) & (cov_arrays['dem'].ravel() > 0)
pred_full = np.full(rows * cols, np.nan, dtype='float32')

pred_full[valid_mask] = rf_final.predict(X_full[valid_mask])
pred_map = pred_full.reshape((rows, cols))

out_pred_path = os.path.join(kovariat_dir, '..', 'prediksi_corganik_0_20cm.tif')
driver = gdal.GetDriverByName('GTiff')
out_ds = driver.Create(out_pred_path, cols, rows, 1, gdal.GDT_Float32, ['COMPRESS=DEFLATE'])
out_ds.SetGeoTransform(gt)
out_ds.SetProjection(ref_ds.GetProjection())
out_b = out_ds.GetRasterBand(1)
out_b.WriteArray(np.where(np.isnan(pred_map), -9999.0, pred_map))
out_b.SetNoDataValue(-9999.0)
out_ds = None

print(f'-> Berhasil menyimpan peta prediksi: {out_pred_path}')
print(f'   Rentang nilai prediksi C-Organik: {np.nanmin(pred_map):.2f}% s.d. {np.nanmax(pred_map):.2f}%')

### 6. Pemodelan Ketidakpastian Spasial (Quantile Regression Forest / QRF)
Kita melatih algoritma QRF untuk memprediksi lebar interval keyakinan 90% (PI90 = $q_{0.95} - q_{0.05}$) untuk estimasi C-Organik.

In [ ]:
# 6. Pemodelan Ketidakpastian QRF (PI90 C-Organik)
qrf = RandomForestQuantileRegressor(n_estimators=300, max_depth=5, random_state=42)
qrf.fit(X, y)

q05 = qrf.predict(X_full[valid_mask], quantiles=[0.05])
q95 = qrf.predict(X_full[valid_mask], quantiles=[0.95])
pi90_valid = q95 - q05

pi90_full = np.full(rows * cols, np.nan, dtype='float32')
pi90_full[valid_mask] = pi90_valid
pi90_map = pi90_full.reshape((rows, cols))

out_unc_path = os.path.join(kovariat_dir, '..', 'ketidakpastian_pi90_corganik.tif')
out_ds = driver.Create(out_unc_path, cols, rows, 1, gdal.GDT_Float32, ['COMPRESS=DEFLATE'])
out_ds.SetGeoTransform(gt)
out_ds.SetProjection(ref_ds.GetProjection())
out_b = out_ds.GetRasterBand(1)
out_b.WriteArray(np.where(np.isnan(pi90_map), -9999.0, pi90_map))
out_b.SetNoDataValue(-9999.0)
out_ds = None

print(f'-> Berhasil menyimpan peta ketidakpastian: {out_unc_path}')
print(f'   Rentang lebar PI90: {np.nanmin(pi90_map):.2f}% s.d. {np.nanmax(pi90_map):.2f}%')

### 7. Visualisasi Hasil Pemodelan Spasial C-Organik
Panel terpadu menampilkan peta prediksi C-Organik, peta ketidakpastian PI90, dan peringkat kepentingan kovariat SCORPAN.

In [ ]:
# 7. Visualisasi Panel DSM C-Organik
fig, axes = plt.subplots(1, 3, figsize=(18, 6.5), dpi=150)

# 1. Peta Prediksi C-Organik
im0 = axes[0].imshow(pred_map, cmap='YlGn', vmin=0.4, vmax=1.5)
axes[0].set_title('Prediksi Spasial C-Organik (%)\nSub-DAS Bompon (Lapisan 0-20 cm)', fontsize=12, fontweight='bold')
axes[0].axis('off')
cbar0 = plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
cbar0.set_label('Kadar C-Organik (%)')

# 2. Peta Ketidakpastian PI90
im1 = axes[1].imshow(pi90_map, cmap='Reds', vmin=0.1, vmax=1.2)
axes[1].set_title('Ketidakpastian Prediksi Spasial (PI90)\nInterval Keyakinan 90% (QRF)', fontsize=12, fontweight='bold')
axes[1].axis('off')
cbar1 = plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
cbar1.set_label('Lebar PI90 C-Organik (%)')

# 3. Peringkat Kepentingan Kovariat
sorted_idx_rev = sorted_idx[::-1]
axes[2].barh(range(len(sorted_idx_rev)), importances[sorted_idx_rev]*100, color='#238b45')
axes[2].set_yticks(range(len(sorted_idx_rev)))
axes[2].set_yticklabels([cov_names[i] for i in sorted_idx_rev], fontsize=10)
axes[2].set_xlabel('Tingkat Kepentingan Relatif (%)', fontsize=11)
axes[2].set_title('Peringkat Kepentingan Kovariat SCORPAN\nPrediktor Karbon Organik Tanah (SOC)', fontsize=12, fontweight='bold')
axes[2].grid(axis='x', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

### Kesimpulan Praktik 5:
Model Random Forest berhasil memetakan variabilitas spasial C-Organik pada resolusi 10 m. Radar SAR Sentinel-1 polarisasi VH dan indeks tanah terbuka BSI terbukti menjadi prediktor paling signifikan, merefleksikan kontribusi biomassa kanopi dan serasah permukaan terhadap kandungan karbon organik tanah.